# Chapitre 4 — Partie 1 : Métriques de Régression

**Durée estimée : 2h**

## 🎯 Objectifs d'apprentissage

À la fin de cette partie, vous serez capable de :
1. **Expliquer** pourquoi l'accuracy ne s'applique pas à la régression
2. **Calculer** et interpréter MSE, RMSE, MAE et R²
3. **Choisir** la métrique appropriée selon le contexte business
4. **Diagnostiquer** la qualité d'un modèle de régression

---

## Lien avec les chapitres précédents

Dans le **Chapitre 2**, nous avons appris à séparer nos données (train/test) et à éviter le data leakage. Dans le **Chapitre 3**, nous avons utilisé `.score()` pour évaluer nos modèles — mais cette méthode nous donnait un seul nombre, sans vraiment expliquer ce qu'il mesurait.

Maintenant, nous allons **ouvrir la boîte noire** des métriques et comprendre exactement ce que signifie "mon modèle est bon".

---

## 🌍 Problème Réel : Combien vaut cette maison ?

Imaginez que vous travaillez pour une agence immobilière. Votre mission : créer un modèle qui prédit le prix des maisons.

Après des semaines de travail, votre modèle est prêt. Votre manager demande : **"Il est bon, ce modèle ?"**

Vous répondez : "Oui, il a un score de 0.85 !"

Votre manager fronce les sourcils : "0.85 quoi ? Euros ? Pourcentage ? Et concrètement, ça veut dire quoi pour nos clients ?"

---

### L'enjeu business des erreurs de prédiction

Selon une [étude comparative 2024 sur la prédiction de prix immobiliers](https://www.emerald.com/insight/content/doi/10.1108/ijhma-09-2023-0120/full/html), les modèles de Machine Learning peuvent avoir des erreurs moyennes significatives :

| Modèle | Erreur typique (RMSE) |
|--------|----------------------|
| Régression linéaire | ~68 600 € |
| Arbre de décision | ~71 400 € |
| Meilleurs modèles optimisés | ~45 000 € |

**Question :** Si votre modèle se trompe en moyenne de 68 600 €, est-ce acceptable ? Comment le savoir ?

*(Réponse attendue : Ça dépend du prix moyen des maisons ! Une erreur de 68 600 € sur des maisons à 500 000 € (~14%) est différente d'une erreur sur des maisons à 150 000 € (~46%))*

---

## 1.1 Pourquoi l'Accuracy ne fonctionne pas pour la régression ?

En **classification**, on prédit des catégories : spam/non-spam, approuvé/refusé. L'accuracy compte simplement le pourcentage de prédictions correctes.

Mais en **régression**, on prédit des valeurs continues : un prix, une température, un chiffre d'affaires...

**Question :** Si le vrai prix est 250 000 € et que vous prédisez 249 999 €, est-ce "correct" ou "incorrect" ?

*(Réponse attendue : C'est presque parfait ! Mais techniquement ce n'est pas exactement égal... L'accuracy binaire ne capture pas cette nuance.)*

```
┌─────────────────────────────────────────────────────────────────────┐
│     CLASSIFICATION vs RÉGRESSION : Le problème de l'accuracy       │
├─────────────────────────────────────────────────────────────────────┤
│                                                                     │
│  CLASSIFICATION (catégories)                                        │
│  ─────────────────────────────                                      │
│  Vrai : "Spam"    Prédit : "Spam"     → ✅ Correct                  │
│  Vrai : "Spam"    Prédit : "Non-spam" → ❌ Incorrect                │
│                                                                     │
│  Accuracy = Nb corrects / Total   (simple et clair)                 │
│                                                                     │
│  ─────────────────────────────────────────────────────────────────  │
│                                                                     │
│  RÉGRESSION (valeurs continues)                                     │
│  ───────────────────────────────                                    │
│  Vrai : 250 000 €   Prédit : 249 999 €  → ✅ ou ❌ ???              │
│  Vrai : 250 000 €   Prédit : 240 000 €  → ✅ ou ❌ ???              │
│  Vrai : 250 000 €   Prédit : 100 000 €  → ✅ ou ❌ ???              │
│                                                                     │
│  → On ne peut pas dire "correct/incorrect"                         │
│  → On doit mesurer "À QUEL POINT" on se trompe                     │
│                                                                     │
└─────────────────────────────────────────────────────────────────────┘
```

**Conclusion :** En régression, on mesure la **distance** entre la prédiction et la réalité, pas un simple "vrai/faux".

---

## 1.2 L'Erreur : La Base de Tout

Avant de voir les métriques, comprenons le concept fondamental : **l'erreur** (ou **résidu**).

```
Erreur = Valeur réelle - Valeur prédite
```

Prenons un exemple concret avec 5 maisons :

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Exemple : prédiction de prix de maisons
donnees = pd.DataFrame({
    'Maison': ['A', 'B', 'C', 'D', 'E'],
    'Prix_reel': [250000, 320000, 180000, 450000, 275000],
    'Prix_predit': [245000, 350000, 175000, 420000, 300000]
})

# Calculer les erreurs
donnees['Erreur'] = donnees['Prix_reel'] - donnees['Prix_predit']

print("📊 Exemple de prédictions de prix immobiliers")
print("=" * 60)
donnees

**Question :** En regardant les erreurs, lesquelles sont "bonnes" et lesquelles sont "mauvaises" ?

*(Réponse attendue : Les erreurs proches de 0 sont bonnes. Mais attention : +5000 et -5000 sont équivalentes en termes de magnitude !)*

In [ ]:
# Visualiser les erreurs
fig, ax = plt.subplots(figsize=(10, 5))

colors = ['green' if abs(e) < 10000 else 'orange' if abs(e) < 30000 else 'red' 
          for e in donnees['Erreur']]

bars = ax.bar(donnees['Maison'], donnees['Erreur'], color=colors, edgecolor='black')
ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
ax.set_ylabel('Erreur (€)')
ax.set_xlabel('Maison')
ax.set_title('Erreurs de prédiction par maison\n(Vert = faible, Rouge = élevée)')

# Ajouter les valeurs
for bar, err in zip(bars, donnees['Erreur']):
    height = bar.get_height()
    ax.annotate(f'{err:+,} €',
                xy=(bar.get_x() + bar.get_width() / 2, height),
                ha='center', va='bottom' if height > 0 else 'top',
                fontsize=10)

plt.tight_layout()
plt.show()

### Le problème des erreurs brutes

Si on fait simplement la **moyenne des erreurs** :

In [ ]:
moyenne_erreurs = donnees['Erreur'].mean()
print(f"Moyenne des erreurs : {moyenne_erreurs:,.0f} €")

**Question :** Cette moyenne semble faible. Est-ce que ça veut dire que notre modèle est excellent ?

*(Réponse attendue : Non ! Les erreurs positives et négatives s'annulent. Une erreur de +30 000 et une de -30 000 donnent une moyenne de 0, mais le modèle se trompe quand même de 30 000 € à chaque fois !)*

---

C'est pour résoudre ce problème qu'on a inventé des métriques plus intelligentes.

---

## 1.3 MAE : Mean Absolute Error (Erreur Absolue Moyenne)

### Construire l'intuition

Le problème des erreurs qui s'annulent a une solution simple : prendre la **valeur absolue** de chaque erreur avant de faire la moyenne.

```
┌─────────────────────────────────────────────────────────────────────┐
│                    MAE : L'IDÉE INTUITIVE                          │
├─────────────────────────────────────────────────────────────────────┤
│                                                                     │
│   Erreurs brutes :     +5 000,  -30 000,  +5 000,  +30 000, -25 000│
│                              ↓                                      │
│   Valeurs absolues :    5 000,   30 000,   5 000,   30 000,  25 000│
│                              ↓                                      │
│   Moyenne :            (5000 + 30000 + 5000 + 30000 + 25000) / 5    │
│                              ↓                                      │
│   MAE = 19 000 €                                                    │
│                                                                     │
│   Interprétation : "En moyenne, le modèle se trompe de 19 000 €"   │
│                                                                     │
└─────────────────────────────────────────────────────────────────────┘
```

In [ ]:
# Calcul manuel du MAE
donnees['Erreur_absolue'] = donnees['Erreur'].abs()
mae_manuel = donnees['Erreur_absolue'].mean()

print("📊 Calcul du MAE (Mean Absolute Error)")
print("=" * 50)
print(donnees[['Maison', 'Erreur', 'Erreur_absolue']])
print(f"\nMAE = {mae_manuel:,.0f} €")
print(f"\n→ En moyenne, le modèle se trompe de {mae_manuel:,.0f} €")

In [ ]:
# Avec scikit-learn
from sklearn.metrics import mean_absolute_error

mae_sklearn = mean_absolute_error(donnees['Prix_reel'], donnees['Prix_predit'])
print(f"MAE (scikit-learn) : {mae_sklearn:,.0f} €")

### La formule mathématique

$$MAE = \frac{1}{n} \sum_{i=1}^{n} |y_i - \hat{y}_i|$$

Où :
- $y_i$ = valeur réelle
- $\hat{y}_i$ = valeur prédite
- $n$ = nombre d'observations

---

┌─────────────────────────────────────────────────────────────────────┐
│ 📖 DÉFINITION : MAE (Mean Absolute Error)                           │
│                                                                     │
│ Le MAE mesure l'erreur moyenne en valeur absolue entre les         │
│ prédictions et les valeurs réelles. C'est la métrique la plus      │
│ intuitive car elle s'exprime dans la même unité que la variable    │
│ cible (euros, degrés, etc.).                                        │
│                                                                     │
│ Caractéristiques :                                                  │
│ • Facile à interpréter : "erreur moyenne de X unités"              │
│ • Robuste aux valeurs extrêmes (outliers)                          │
│ • Toutes les erreurs ont le même poids                              │
│                                                                     │
│ Plus le MAE est proche de 0, meilleur est le modèle.               │
└─────────────────────────────────────────────────────────────────────┘

---

## 1.4 MSE et RMSE : Pénaliser les Grosses Erreurs

### Le problème du MAE

Avec le MAE, une erreur de 1 000 € compte autant que... 1 000 €. Logique, non ?

Mais dans certains contextes, une **grosse erreur** est bien plus grave que plusieurs petites.

**Question :** Que préférez-vous ?
- **Option A** : 10 erreurs de 5 000 € chacune (MAE = 5 000 €)
- **Option B** : 9 erreurs de 0 € et 1 erreur de 50 000 € (MAE = 5 000 €)

*(Réponse attendue : Probablement l'option A ! Une seule erreur catastrophique de 50 000 € peut ruiner une vente ou une relation client, même si la moyenne est identique.)*

### L'idée : mettre au carré

En mettant les erreurs **au carré**, on donne plus de poids aux grosses erreurs :

```
┌─────────────────────────────────────────────────────────────────────┐
│              EFFET DU CARRÉ SUR LES ERREURS                        │
├─────────────────────────────────────────────────────────────────────┤
│                                                                     │
│   Erreur    →   Erreur²                                             │
│   ───────       ────────                                            │
│   1 000 €   →   1 000 000          (×1 000)                         │
│   5 000 €   →   25 000 000         (×5 000)                         │
│   10 000 €  →   100 000 000        (×10 000)                        │
│   50 000 €  →   2 500 000 000      (×50 000)                        │
│                                                                     │
│   → Une erreur 10× plus grande devient 100× plus pénalisée !       │
│                                                                     │
└─────────────────────────────────────────────────────────────────────┘
```

In [ ]:
# MSE : Mean Squared Error
donnees['Erreur_carree'] = donnees['Erreur'] ** 2
mse_manuel = donnees['Erreur_carree'].mean()

print("📊 Calcul du MSE (Mean Squared Error)")
print("=" * 60)
print(donnees[['Maison', 'Erreur', 'Erreur_carree']])
print(f"\nMSE = {mse_manuel:,.0f}")

**Problème :** Le MSE est en "euros au carré" — pas très parlant !

**Solution :** Prendre la **racine carrée** pour revenir à l'unité d'origine → **RMSE**

In [ ]:
# RMSE : Root Mean Squared Error
rmse_manuel = np.sqrt(mse_manuel)

print(f"RMSE = √MSE = √{mse_manuel:,.0f} = {rmse_manuel:,.0f} €")
print(f"\n→ Erreur 'typique' du modèle : {rmse_manuel:,.0f} €")

In [ ]:
# Avec scikit-learn
from sklearn.metrics import mean_squared_error

mse_sklearn = mean_squared_error(donnees['Prix_reel'], donnees['Prix_predit'])
rmse_sklearn = np.sqrt(mse_sklearn)  # ou mean_squared_error(..., squared=False)

print(f"MSE (scikit-learn)  : {mse_sklearn:,.0f}")
print(f"RMSE (scikit-learn) : {rmse_sklearn:,.0f} €")

### Comparer MAE et RMSE

In [ ]:
print("📊 Comparaison MAE vs RMSE")
print("=" * 40)
print(f"MAE  = {mae_manuel:,.0f} €")
print(f"RMSE = {rmse_manuel:,.0f} €")
print(f"\nDifférence : {rmse_manuel - mae_manuel:,.0f} €")
print(f"\n💡 RMSE > MAE indique la présence de grosses erreurs")

<details>
<summary>🤔 Question Socratique : Pourquoi le RMSE est-il toujours ≥ MAE ?</summary>

### 🔑 Réponse

C'est une propriété mathématique : la moyenne quadratique est toujours supérieure ou égale à la moyenne arithmétique.

**Intuition :** Le carré "amplifie" les grosses erreurs plus que les petites. Quand on fait la moyenne puis la racine, cette amplification persiste.

**Cas d'égalité :** RMSE = MAE uniquement si **toutes les erreurs sont identiques** (en valeur absolue).

**Indication pratique :**
- Si RMSE ≈ MAE → Erreurs uniformes
- Si RMSE >> MAE → Présence de quelques grosses erreurs (outliers)

</details>

---

┌─────────────────────────────────────────────────────────────────────┐
│ 📖 DÉFINITION : MSE et RMSE                                         │
│                                                                     │
│ **MSE (Mean Squared Error)** : Moyenne des erreurs au carré.       │
│ Formule : MSE = (1/n) × Σ(yᵢ - ŷᵢ)²                                │
│                                                                     │
│ **RMSE (Root Mean Squared Error)** : Racine carrée du MSE.         │
│ Formule : RMSE = √MSE                                               │
│                                                                     │
│ Caractéristiques :                                                  │
│ • Pénalise fortement les grosses erreurs                           │
│ • RMSE s'exprime dans l'unité d'origine (euros, etc.)              │
│ • MSE est utilisé pour l'optimisation (dérivable)                  │
│ • Sensible aux outliers                                             │
│                                                                     │
│ Plus le RMSE est proche de 0, meilleur est le modèle.              │
└─────────────────────────────────────────────────────────────────────┘

---

## 1.5 R² : Le Coefficient de Détermination

### Le problème des métriques absolues

MAE et RMSE donnent des valeurs en euros. Mais :

- Une RMSE de 20 000 € est-elle bonne ou mauvaise ?
- Comment comparer deux modèles sur des datasets différents ?

**Question :** Si je vous dis "mon modèle a une RMSE de 50 000 €", pouvez-vous me dire s'il est bon ?

*(Réponse attendue : Impossible à dire sans contexte ! 50 000 € d'erreur sur des maisons à 2 millions est excellent, sur des studios à 80 000 € c'est catastrophique.)*

### L'idée du R² : comparer à un modèle "naïf"

Le R² répond à la question : **"Mon modèle est-il meilleur que de simplement prédire la moyenne ?"**

```
┌─────────────────────────────────────────────────────────────────────┐
│              R² : COMPARAISON AVEC LA MOYENNE                       │
├─────────────────────────────────────────────────────────────────────┤
│                                                                     │
│   Modèle "naïf" : toujours prédire la moyenne des prix             │
│   (Comme le DummyRegressor de sklearn)                              │
│                                                                     │
│   Prix réels : 250k, 320k, 180k, 450k, 275k                         │
│   Moyenne = 295k                                                    │
│                                                                     │
│   Modèle naïf prédit : 295k, 295k, 295k, 295k, 295k                 │
│   → Erreur totale du naïf = SST (Total Sum of Squares)             │
│                                                                     │
│   Votre modèle prédit : 245k, 350k, 175k, 420k, 300k                │
│   → Erreur de votre modèle = SSE (Sum of Squared Errors)           │
│                                                                     │
│   R² = 1 - (SSE / SST)                                              │
│                                                                     │
│   → R² = 1 : Modèle parfait (SSE = 0)                              │
│   → R² = 0 : Modèle = moyenne (SSE = SST)                          │
│   → R² < 0 : Modèle PIRE que la moyenne !                          │
│                                                                     │
└─────────────────────────────────────────────────────────────────────┘
```

In [ ]:
# Calcul manuel du R²
y_reel = donnees['Prix_reel']
y_pred = donnees['Prix_predit']
y_moyenne = y_reel.mean()

# SSE : erreur de notre modèle
SSE = ((y_reel - y_pred) ** 2).sum()

# SST : erreur du modèle naïf (moyenne)
SST = ((y_reel - y_moyenne) ** 2).sum()

# R²
r2_manuel = 1 - (SSE / SST)

print("📊 Calcul du R² (Coefficient de détermination)")
print("=" * 50)
print(f"Moyenne des prix : {y_moyenne:,.0f} €")
print(f"\nSSE (erreur modèle)  : {SSE:,.0f}")
print(f"SST (erreur moyenne) : {SST:,.0f}")
print(f"\nR² = 1 - ({SSE:,.0f} / {SST:,.0f})")
print(f"R² = {r2_manuel:.4f}")

In [ ]:
# Avec scikit-learn
from sklearn.metrics import r2_score

r2_sklearn = r2_score(y_reel, y_pred)
print(f"R² (scikit-learn) : {r2_sklearn:.4f}")
print(f"\n→ Le modèle explique {r2_sklearn*100:.1f}% de la variance des prix")

### Comment interpréter le R² ?

In [ ]:
print("""
┌─────────────────────────────────────────────────────────────────────┐
│              INTERPRÉTATION DU R²                                   │
├─────────────────────────────────────────────────────────────────────┤
│                                                                     │
│   R² = 1.00     Parfait (impossible en pratique)                   │
│   R² > 0.90     Excellent                                           │
│   R² > 0.70     Bon                                                 │
│   R² > 0.50     Acceptable                                          │
│   R² > 0.30     Faible mais peut être utile                        │
│   R² ≈ 0.00     Modèle = moyenne (inutile)                         │
│   R² < 0.00     Pire que la moyenne ! 🚨                           │
│                                                                     │
│   ⚠️ ATTENTION : Ces seuils dépendent du domaine !                  │
│   • Finance/Bourse : R² = 0.10 peut être excellent                 │
│   • Physique : R² < 0.99 peut être insuffisant                     │
│                                                                     │
└─────────────────────────────────────────────────────────────────────┘
""")

<details>
<summary>🤔 Question Socratique : Un R² négatif est-il vraiment possible ? Que signifie-t-il ?</summary>

### 🔑 Réponse

Oui, c'est possible ! Un R² négatif signifie que votre modèle fait **pire** que simplement prédire la moyenne.

**Causes possibles :**
- Modèle mal entraîné ou mal configuré
- Features non pertinentes
- Évaluation sur des données très différentes de l'entraînement
- Overfitting sévère

**Action :** Si R² < 0 sur le test set, c'est une alerte rouge ! Vérifiez votre pipeline.

</details>

---

┌─────────────────────────────────────────────────────────────────────┐
│ 📖 DÉFINITION : R² (Coefficient de Détermination)                   │
│                                                                     │
│ Le R² mesure la proportion de variance de la variable cible        │
│ expliquée par le modèle. C'est une métrique relative qui compare   │
│ les performances du modèle à un modèle naïf (la moyenne).          │
│                                                                     │
│ Formule : R² = 1 - (SSE / SST)                                     │
│           où SSE = Σ(yᵢ - ŷᵢ)² et SST = Σ(yᵢ - ȳ)²                │
│                                                                     │
│ Caractéristiques :                                                  │
│ • Valeur entre -∞ et 1 (idéalement entre 0 et 1)                   │
│ • Sans unité → comparable entre datasets                           │
│ • Interprétation : "% de variance expliquée"                       │
│                                                                     │
│ C'est la métrique retournée par .score() sur les régresseurs.      │
└─────────────────────────────────────────────────────────────────────┘

---

## 1.6 Récapitulatif : Quelle Métrique Choisir ?

```
┌─────────────────────────────────────────────────────────────────────┐
│           GUIDE DE SÉLECTION DES MÉTRIQUES DE RÉGRESSION           │
├─────────────────────────────────────────────────────────────────────┤
│                                                                     │
│   MÉTRIQUE │ QUAND L'UTILISER                                       │
│   ─────────┼───────────────────────────────────────────────────────│
│            │                                                        │
│   MAE      │ • Vous voulez une interprétation simple               │
│            │ • Les grosses erreurs ne sont pas plus graves         │
│            │ • Vous avez des outliers dans vos données             │
│            │ • Communication avec des non-techniques               │
│            │                                                        │
│   RMSE     │ • Les grosses erreurs sont particulièrement graves    │
│            │ • Vous optimisez un modèle (gradient descent)         │
│            │ • Standard dans les compétitions ML                   │
│            │                                                        │
│   R²       │ • Vous comparez des modèles sur différents datasets   │
│            │ • Vous voulez savoir si le modèle apporte de la valeur│
│            │ • Communication du "pouvoir explicatif"               │
│            │                                                        │
│   ─────────┼───────────────────────────────────────────────────────│
│            │                                                        │
│   💡 CONSEIL : Utilisez PLUSIEURS métriques ensemble !             │
│      R² pour la vue d'ensemble + RMSE/MAE pour l'erreur concrète   │
│                                                                     │
└─────────────────────────────────────────────────────────────────────┘
```

In [ ]:
# Fonction utilitaire pour afficher toutes les métriques
def evaluer_regression(y_vrai, y_pred, nom_modele="Modèle"):
    """Affiche toutes les métriques de régression importantes."""
    from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
    
    mae = mean_absolute_error(y_vrai, y_pred)
    mse = mean_squared_error(y_vrai, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_vrai, y_pred)
    
    print(f"\n📊 Évaluation : {nom_modele}")
    print("=" * 50)
    print(f"MAE  : {mae:,.2f}")
    print(f"RMSE : {rmse:,.2f}")
    print(f"R²   : {r2:.4f} ({r2*100:.1f}% de variance expliquée)")
    
    return {'MAE': mae, 'RMSE': rmse, 'R2': r2}

# Test
evaluer_regression(donnees['Prix_reel'], donnees['Prix_predit'], "Notre modèle immobilier")

---

## 🧪 Exercice Pratique : Évaluer un Modèle de Prédiction de Salaires

Vous travaillez pour un cabinet RH. On vous demande d'évaluer un modèle qui prédit les salaires.

In [ ]:
# Données de l'exercice
np.random.seed(42)

exercice = pd.DataFrame({
    'Employe': [f'E{i}' for i in range(1, 11)],
    'Salaire_reel': [45000, 52000, 38000, 75000, 62000, 48000, 55000, 42000, 68000, 58000],
    'Salaire_predit': [47000, 49000, 40000, 70000, 65000, 46000, 58000, 38000, 72000, 55000]
})

print("📊 Données de prédiction de salaires")
print("=" * 50)
exercice

### Votre mission :

1. Calculez le MAE, RMSE et R² (avec scikit-learn)
2. Interprétez chaque métrique en langage business
3. Répondez : ce modèle est-il suffisamment bon pour être déployé ?

In [ ]:
# 🎯 À VOUS DE JOUER !

# Étape 1 : Calculer les métriques
# ...

# Étape 2 : Interpréter
# ...

# Étape 3 : Recommandation
# ...

### 🔑 Solution

In [ ]:
# Solution - Étape 1 : Calculer les métriques
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

y_vrai = exercice['Salaire_reel']
y_pred = exercice['Salaire_predit']

mae = mean_absolute_error(y_vrai, y_pred)
rmse = np.sqrt(mean_squared_error(y_vrai, y_pred))
r2 = r2_score(y_vrai, y_pred)

print("📊 Résultats")
print("=" * 50)
print(f"MAE  : {mae:,.0f} €")
print(f"RMSE : {rmse:,.0f} €")
print(f"R²   : {r2:.4f}")

In [ ]:
# Solution - Étape 2 : Interpréter
salaire_moyen = y_vrai.mean()
erreur_relative = (mae / salaire_moyen) * 100

print("\n📋 Interprétation business")
print("=" * 50)
print(f"\n• MAE = {mae:,.0f} €")
print(f"  → En moyenne, le modèle se trompe de {mae:,.0f} € par employé")
print(f"  → Sur un salaire moyen de {salaire_moyen:,.0f} €, c'est {erreur_relative:.1f}% d'erreur")

print(f"\n• RMSE = {rmse:,.0f} €")
print(f"  → RMSE proche du MAE = pas de grosses erreurs isolées ✓")

print(f"\n• R² = {r2:.4f}")
print(f"  → Le modèle explique {r2*100:.1f}% de la variance des salaires")
print(f"  → C'est {'bon' if r2 > 0.7 else 'acceptable' if r2 > 0.5 else 'faible'}")

In [ ]:
# Solution - Étape 3 : Recommandation
print("\n💼 Recommandation")
print("=" * 50)

if r2 > 0.8 and erreur_relative < 5:
    print("✅ Modèle prêt pour le déploiement")
elif r2 > 0.7 and erreur_relative < 10:
    print("⚠️ Modèle acceptable mais perfectible")
    print("   → Peut être utilisé avec supervision humaine")
else:
    print("❌ Modèle à améliorer avant déploiement")
    print("   → Erreur trop importante pour des décisions RH")

print(f"\n   Une erreur de {mae:,.0f} € peut impacter :")
print(f"   - Les négociations salariales")
print(f"   - L'équité interne")
print(f"   - Le budget RH")

---

## 🧠 Réflexion Métacognitive

Avant de passer à la suite :

1. **Pouvez-vous expliquer** la différence entre MAE et RMSE à un collègue non-technique ?

2. **Dans quel cas** préféreriez-vous utiliser le MAE plutôt que le RMSE ?

3. **Un R² de 0.60** est-il bon ou mauvais ? (Réponse : ça dépend du contexte !)

---

## 📝 Résumé

| Métrique | Formule | Interprétation | Sensibilité outliers |
|----------|---------|----------------|---------------------|
| **MAE** | Moyenne des \|erreurs\| | Erreur moyenne en unités | Faible |
| **MSE** | Moyenne des erreurs² | Pour l'optimisation | Forte |
| **RMSE** | √MSE | Erreur "typique" en unités | Forte |
| **R²** | 1 - SSE/SST | % de variance expliquée | Moyenne |

**Code essentiel :**
```python
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

mae = mean_absolute_error(y_vrai, y_pred)
rmse = np.sqrt(mean_squared_error(y_vrai, y_pred))
r2 = r2_score(y_vrai, y_pred)
```

---

## ➡️ Prochaine partie

Dans la **Partie 2 : Métriques de Classification**, nous allons explorer en profondeur precision, recall, F1-score et la courbe ROC-AUC.

**Question de transition :** Si votre modèle de détection de fraude a 99% d'accuracy mais ne détecte aucune fraude, est-il vraiment bon ?